# India Runs - Intelligent Candidate Discovery & Ranking Validation

This notebook provides exploratory analysis, validation, and metrics for our rules-based candidate ranking system for the **Senior AI Engineer — Founding Team** role at **Redrob AI**.

## 1. Load Dataset and Libraries
We load standard libraries (json, math, datetime, collections) and candidate profiles from `candidates.jsonl`.

In [1]:
import json
import math
import os
from datetime import datetime
from collections import Counter, defaultdict

candidates_path = '../data/candidates.jsonl'
print("Loading candidates dataset...")
candidates = []
with open(candidates_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            candidates.append(json.loads(line))
print(f"Loaded {len(candidates)} candidates successfully.")

Loading candidates dataset...
Loaded 100000 candidates successfully.


## 2. Dynamic Semantic Matching (Token Similarity)
To handle semantic matching contextually (avoiding strict hardcoded lists), we compute substring matches and word token overlap Jaccard similarity. This allows us to score related terms (e.g. "ColBERT", "dense passage retrieval") naturally.

In [2]:
# Extract target keywords from the Job Description
jd_keywords = {
    'rag', 'retrieval-augmented generation', 'vector', 'database', 'embeddings',
    'sentence-transformers', 'transformers', 'pinecone', 'weaviate', 'qdrant',
    'milvus', 'faiss', 'elasticsearch', 'opensearch', 'search', 'retrieval',
    'semantic', 'matching', 'ranking', 'recommendation', 'nlp', 'language',
    'ndcg', 'map', 'mrr', 'evaluation', 'xgboost', 'lora', 'qlora', 'peft',
    'fine-tuning', 'python', 'backend', 'system design'
}
similarity_cache = {}
def calculate_token_similarity(term, targets):
    term_lower = term.lower().strip()
    if term_lower in similarity_cache:
        return similarity_cache[term_lower]
    best_sim = 0.0
    
    for target in targets:
        # 1. Exact or substring check
        if term_lower == target or target in term_lower or term_lower in target:
            sim = 1.0
        else:
            # 2. Token overlap Jaccard similarity
            term_tokens = set(term_lower.split())
            target_tokens = set(target.split())
            if term_tokens and target_tokens:
                intersect = term_tokens.intersection(target_tokens)
                union = term_tokens.union(target_tokens)
                sim = len(intersect) / len(union)
            else:
                sim = 0.0
        
        # 3. Simple character n-gram fallback for fuzzy matching (length 3)
        if sim < 0.4 and len(term_lower) >= 3 and len(target) >= 3:
            t_grams = set(term_lower[i:i+3] for i in range(len(term_lower)-2))
            trg_grams = set(target[i:i+3] for i in range(len(target)-2))
            char_sim = len(t_grams.intersection(trg_grams)) / len(t_grams.union(trg_grams))
            sim = max(sim, char_sim)
            
        best_sim = max(best_sim, sim)
    similarity_cache[term_lower] = best_sim
    return best_sim
# Test similarity
test_terms = ['ColBERT', 'dense retrieval', 'hybrid search', 'java developer', 'marketing specialist']
for term in test_terms:
    sim = calculate_token_similarity(term, jd_keywords)
    print(f"Term: {term:<25} | Semantic Match Score: {sim:.3f}")



Term: ColBERT                   | Semantic Match Score: 0.000
Term: dense retrieval           | Semantic Match Score: 1.000
Term: hybrid search             | Semantic Match Score: 1.000
Term: java developer            | Semantic Match Score: 0.053
Term: marketing specialist      | Semantic Match Score: 0.045


## 3. Anomaly & Honeypot Detection (Tightening Rules)
We run tighter logic to detect logically impossible candidate profiles (honeypots), targeting the ~80 profiles mentioned in the README.

In [3]:
founding_years = {
    'Krutrim': 2023,
    'Sarvam AI': 2023,
    'CRED': 2018,
    'Glance': 2019,
    'Meesho': 2015,
    'PhonePe': 2015,
    'PharmEasy': 2015,
    'Swiggy': 2014,
    'Ola': 2010,
    'Zomato': 2008,
    'Flipkart': 2007,
    'Freshworks': 2010,
    'InMobi': 2007,
    'Zoho': 1996
}

def is_honeypot(c):
    prof = c.get('profile', {})
    hist = c.get('career_history', [])
    edu = c.get('education', [])
    skills = c.get('skills', [])
    
    # A: skills duration is 0 but proficiency is expert/advanced (threshold: >= 5 skills)
    z_skills = sum(1 for s in skills if s.get('proficiency') in ['expert', 'advanced'] and s.get('duration_months', 0) == 0)
    if z_skills >= 5:
        return True
        
    # B: job duration mismatch (inflated duration: reported exceeds calendar range by factor of 2 + 12 months)
    for h in hist:
        sd = h.get('start_date')
        ed = h.get('end_date')
        dur = h.get('duration_months', 0)
        if sd:
            try:
                s_dt = datetime.strptime(sd, "%Y-%m-%d")
                if ed:
                    e_dt = datetime.strptime(ed, "%Y-%m-%d")
                else:
                    e_dt = datetime.strptime("2026-06-20", "%Y-%m-%d")
                diff_months = (e_dt.year - s_dt.year) * 12 + (e_dt.month - s_dt.month)
                if dur > 2 * diff_months + 12:
                    return True
            except:
                pass
                
    # C: education start/end reversed
    for e in edu:
        sy = e.get('start_year')
        ey = e.get('end_year')
        if sy and ey and sy > ey:
            return True
            
    # D: work starts way before college (e.g. started full-time job 15 years before starting college)
    edu_starts = [e.get('start_year') for e in edu if e.get('start_year')]
    if edu_starts:
        min_edu_start = min(edu_starts)
        for h in hist:
            sd = h.get('start_date')
            if sd:
                try:
                    start_yr = int(sd.split('-')[0])
                    if min_edu_start - start_yr > 15:
                        return True
                except:
                    pass
                    
    # E: founding year violation
    for h in hist:
        comp = h.get('company')
        sd = h.get('start_date')
        if comp in founding_years and sd:
            try:
                start_yr = int(sd.split('-')[0])
                if start_yr < founding_years[comp]:
                    return True
            except:
                pass
                
    return False

honeypots_detected = [c['candidate_id'] for c in candidates if is_honeypot(c)]
print(f"Detected {len(honeypots_detected)} honeypots in total candidate pool.")

Detected 274 honeypots in total candidate pool.


## 4. Continuous Consulting Penalty
We calculate the fraction of career spent at IT consulting/services firms and apply a smooth multiplier: `1.0 - 0.5 * consulting_ratio`.

In [4]:
consulting_companies = {
    'TCS', 'Infosys', 'Wipro', 'Accenture', 'Cognizant', 'Capgemini',
    'Tech Mahindra', 'Mphasis', 'HCL', 'Mindtree', 'Genpact AI',
    'Deloitte', 'PwC', 'EY', 'KPMG'
}

def calculate_consulting_ratio(hist):
    if not hist:
        return 0.0
    consulting_months = 0
    total_months = 0
    for h in hist:
        dur = h.get('duration_months', 0)
        comp = h.get('company')
        total_months += dur
        if comp in consulting_companies:
            consulting_months += dur
    if total_months == 0:
        return 0.0
    return consulting_months / total_months

# Test ratios
sample_hist_consulting = [{'company': 'TCS', 'duration_months': 48}, {'company': 'Infosys', 'duration_months': 24}]
sample_hist_hybrid = [{'company': 'TCS', 'duration_months': 36}, {'company': 'Hooli', 'duration_months': 36}]
sample_hist_product = [{'company': 'Google', 'duration_months': 60}]

print(f"Consulting-only ratio: {calculate_consulting_ratio(sample_hist_consulting):.2%}")
print(f"Hybrid career ratio: {calculate_consulting_ratio(sample_hist_hybrid):.2%}")
print(f"Product career ratio: {calculate_consulting_ratio(sample_hist_product):.2%}")

Consulting-only ratio: 100.00%
Hybrid career ratio: 50.00%
Product career ratio: 0.00%


## 5. Normalized Scoring Curve & Interpretability
The relevance score is a sum of four components: Experience Fit (25 pts), Skill Match (40 pts), Title Fit (25 pts), and Location (10 pts), summing up to 100.0.

In [5]:
def get_yoe_fit(yoe):
    # Peaks at 5-9 years experience
    if 5.0 <= yoe <= 9.0:
        return 25.0
    elif yoe < 5.0:
        return max(0.0, 25.0 - (5.0 - yoe) * 5.0)
    else:
        return max(0.0, 25.0 - (yoe - 9.0) * 1.5)

def get_skills_fit(skills, targets):
    if not skills:
        return 0.0
    total_match = 0.0
    for s in skills:
        name = s.get('name', '')
        prof = s.get('proficiency', 'beginner')
        dur = s.get('duration_months', 0)
        
        sim = calculate_token_similarity(name, targets)
        if sim > 0.4:
            prof_mult = 1.2 if prof == 'expert' else (1.0 if prof == 'advanced' else (0.8 if prof == 'intermediate' else 0.5))
            dur_mult = min(1.2, math.log(dur + 1) / math.log(60)) if dur > 0 else 1.0
            total_match += sim * prof_mult * dur_mult
    return min(40.0, total_match * 8.0) # Normalized to max 40 points

def get_title_fit(prof, hist, targets):
    title = prof.get('current_title', '')
    headline = prof.get('headline', '')
    
    # Verify current role fit
    cur_sim = max(calculate_token_similarity(title, targets), calculate_token_similarity(headline, targets))
    
    # Verify history descriptions matching
    desc_matches = 0
    seen = set()
    for h in hist:
        desc = h.get('description', '')
        for t in targets:
            if t in desc.lower() and t not in seen:
                desc_matches += 1
                seen.add(t)
                
    hist_score = min(10.0, desc_matches * 2.5)
    return min(25.0, cur_sim * 15.0 + hist_score)

def get_location_fit(prof):
    country = prof.get('country', '').lower().strip()
    loc = prof.get('location', '').lower()
    willing = prof.get('willing_to_relocate', False) or (prof.get('willing_to_relocate') == 'true')
    
    is_india = 'india' in country or loc in ['pune', 'noida', 'delhi ncr', 'mumbai', 'hyderabad', 'gurgaon', 'bangalore', 'chennai']
    tier1_cities = ['pune', 'noida', 'delhi', 'ncr', 'mumbai', 'hyderabad', 'gurgaon', 'bangalore', 'chennai', 'kolkata']
    is_tier1 = any(city in loc for city in tier1_cities)
    
    if is_india:
        return 10.0 if is_tier1 else (8.0 if willing else 2.0)
    return 4.0 if willing else 0.0

## 6. Safe Behavioral Modifier
Instead of compounding independent multipliers which lead to extreme unclipped swings, we use a weighted average of individual behavioral signals and clip the combined multiplier to `[0.5, 1.2]`.

In [6]:
def calculate_behavioral_modifier(sigs):
    scores = []
    
    # 1. Recruiter Response Rate (weight 3)
    rrr = sigs.get('recruiter_response_rate', 0.0)
    scores.append((rrr, 3))
    
    # 2. Platform Activity (weight 2)
    last_act_str = sigs.get('last_active_date', '')
    act_score = 1.0
    if last_act_str:
        try:
            last_act = datetime.strptime(last_act_str, "%Y-%m-%d")
            curr_date = datetime(2026, 6, 20)
            days_inactive = (curr_date - last_act).days
            # Fixed dead-code check ordering
            if days_inactive > 360:
                act_score = 0.5
            elif days_inactive > 180:
                act_score = 0.7
        except:
            pass
    scores.append((act_score, 2))
    
    # 3. Notice Period (weight 2)
    notice = sigs.get('notice_period_days', 90)
    notice_score = 1.1 if notice <= 30 else (1.0 if notice <= 60 else (0.8 if notice <= 90 else 0.5))
    scores.append((notice_score, 2))
    
    # 4. Open to Work Flag (weight 1)
    otw = 1.1 if sigs.get('open_to_work_flag', False) else 0.95
    scores.append((otw, 1))
    
    # Calculate weighted average
    total_score = sum(val * wt for val, wt in scores)
    total_wt = sum(wt for val, wt in scores)
    avg_score = total_score / total_wt
    
    # Clip to [0.5, 1.2]
    return max(0.5, min(1.2, avg_score))

## 7. Evaluate and Rank Candidates
We evaluate all candidates, sort them, and print the top 10 ranked profiles.

In [7]:
import time
start_time = time.time()
ranked_list = []
for c in candidates:
    if is_honeypot(c):
        score = -999.0
    else:
        prof = c.get('profile', {})
        hist = c.get('career_history', [])
        skills = c.get('skills', [])
        sigs = c.get('redrob_signals', {})
        
        yoe = prof.get('years_of_experience', 0.0)
        
        y_score = get_yoe_fit(yoe)
        s_score = get_skills_fit(skills, jd_keywords)
        t_score = get_title_fit(prof, hist, jd_keywords)
        l_score = get_location_fit(prof)
        
        raw_score = y_score + s_score + t_score + l_score
        
        # Consulting modifier
        c_ratio = calculate_consulting_ratio(hist)
        c_mult = 1.0 - 0.5 * c_ratio
        
        # Behavioral modifier
        b_mult = calculate_behavioral_modifier(sigs)
        
        score = raw_score * c_mult * b_mult
        
    ranked_list.append((score, c))

ranked_list.sort(key=lambda x: (-x[0], x[1]['candidate_id']))

print("TOP 10 CANDIDATES SCORING:")
for i in range(10):
    score, c = ranked_list[i]
    prof = c['profile']
    sigs = c['redrob_signals']
    print(f"#{i+1}: ID: {c['candidate_id']} | Score: {score:.3f} | Name: {prof['anonymized_name']} | Title: {prof['current_title']} | YoE: {prof['years_of_experience']} | Active: {sigs['last_active_date']}")
print(f"Evaluation completed in {time.time() - start_time:.2f} seconds.")



TOP 10 CANDIDATES SCORING:
#1: ID: CAND_0064326 | Score: 99.000 | Name: Nisha Pillai | Title: Search Engineer | YoE: 7.6 | Active: 2026-05-21
#2: ID: CAND_0068351 | Score: 96.625 | Name: Aadhya Iyer | Title: Lead AI Engineer | YoE: 6.4 | Active: 2026-04-11
#3: ID: CAND_0052328 | Score: 95.875 | Name: Vikram Banerjee | Title: Recommendation Systems Engineer | YoE: 6.5 | Active: 2026-03-31
#4: ID: CAND_0000031 | Score: 95.428 | Name: Ela Singh | Title: Recommendation Systems Engineer | YoE: 6.0 | Active: 2026-05-24
#5: ID: CAND_0005260 | Score: 94.125 | Name: Mira Ghosh | Title: Senior NLP Engineer | YoE: 5.2 | Active: 2026-05-10
#6: ID: CAND_0065195 | Score: 93.750 | Name: Kiara Mukherjee | Title: Search Engineer | YoE: 5.1 | Active: 2026-05-24
#7: ID: CAND_0006418 | Score: 93.338 | Name: Rahul Mukherjee | Title: Machine Learning Engineer | YoE: 5.7 | Active: 2026-03-31
#8: ID: CAND_0054123 | Score: 93.082 | Name: Tanvi Banerjee | Title: Applied ML Engineer | YoE: 4.7 | Active: 2026-05-